# 8.1 CANN 编程错误调试

## 前置要求

具备 C++、Ascend C、CMake 和简单 Vector Kernel 构建经验；使用带 `mssanitizer`、`msdebug` 的 CANN 9.0 环境。

## 章节目标

- 把四类运行期异常定位到源码；
- 区分独立 ASC 与标准算子工程的插桩位置；
- 使用 msDebug 的断点、调用栈、栈帧和源码查看命令；
- 区分 `PASS`、`EXPECTED_DIAGNOSTIC` 与 `BLOCKED`。

本章按“故障代码 → 构建形态 → 工具命令 → 诊断特征 → 根因 → 修复验证”形成闭环。故障程序出现预期诊断不是正常程序 PASS；权限不足也不是 FAIL 或 PASS，而是 `BLOCKED`。

<img src="images/debug_fault_tool_matrix.svg" width="760" style="display:block; margin-left:0;" />


<table style="text-align:left; margin-left:0;">
<tr><th>工具</th><th>代码问题</th><th>关键诊断</th><th>实验状态</th></tr>
<tr><td>memcheck</td><td>DataCopy 长度超过 LocalTensor</td><td>illegal read/write</td><td>EXPECTED_DIAGNOSTIC</td></tr>
<tr><td>racecheck</td><td>MTE2 到 Vector 缺少 WaitFlag</td><td>RAW hazard</td><td>EXPECTED_DIAGNOSTIC</td></tr>
<tr><td>initcheck</td><td>未初始化 LocalTensor 被搬出</td><td>uninitialized read</td><td>EXPECTED_DIAGNOSTIC</td></tr>
<tr><td>synccheck</td><td>SetFlag 没有匹配 WaitFlag</td><td>unpaired set_flag</td><td>EXPECTED_DIAGNOSTIC</td></tr>
<tr><td>msDebug</td><td>观察 Kernel 执行位置</td><td>breakpoint/backtrace/frame/source</td><td>PASS 或 BLOCKED</td></tr>
</table>

课程永不执行 `sudo`、`chmod`、`chown`，也不写 `/proc/debug_switch`。这些动作只能由环境管理员在 Notebook 外完成。


In [ ]:
from pathlib import Path
import os
import shutil

print("mssanitizer:", shutil.which("mssanitizer") or "missing")
print("msdebug:", shutil.which("msdebug") or "missing")
print("fault source:", Path("src/add_sanitizer.asc").is_file())

switch = Path("/proc/debug_switch")
print("debug switch:", switch.read_text().strip() if switch.is_file() and os.access(switch, os.R_OK) else "missing/unreadable")
drv_debug = Path("/dev/drv_debug")
print("/dev/drv_debug present:", drv_debug.exists())
print("/dev/drv_debug readable/writable:", os.access(drv_debug, os.R_OK | os.W_OK) if drv_debug.exists() else False)


## 章节内容

<table style="text-align:left; margin-left:0;">
<tr><th>小节</th><th>内容</th><th>入口</th></tr>
<tr><td>8.2</td><td>两种工程形态的四类诊断、预期退出处理与 msDebug 权限门</td><td><a href="./08.02_msdebug_mssanitizer.ipynb">08.02_msdebug_mssanitizer.ipynb</a></td></tr>
<tr><td>8.3</td><td>客观题与简单/中等/困难三档实践</td><td><a href="./08.03_chapter_test.ipynb">08.03_chapter_test.ipynb</a></td></tr>
</table>
